# 深市 15:00 后逐笔记录核查
口径：TransactTime > 15:00:00.000；等于边界单独统计。按 Parquet 行组时间统计剪枝，读取全部可能命中行组。股票/ETF 前缀与项目保持一致；非零分布不包括无命中证券。LocalTime 不用于边界判定。


In [ ]:
from collections import Counter
from pathlib import Path
import json
import pyarrow.compute as pc
import pyarrow.parquet as pq

ROOT = Path("/hdd/data/stock/raw_level2_parquet")
CUTOFF = "15:00:00.000"
DATES = ["20260803", "20260814", "20260827", "20260828"]

def category(symbol):
    if len(symbol) == 6 and symbol.isdigit():
        if symbol.startswith(("000", "001", "002", "003", "300", "301", "302")):
            return "stock"
        if symbol.startswith("159"):
            return "etf"
    return "other"

results = []
for day in DATES:
    result = {"date": day, "files": {}}
    for dataset in ["mdl_6_33_0", "mdl_6_36_0"]:
        path = ROOT / ("date=" + day) / dataset / "part-0.parquet"
        if not path.is_file():
            result["files"][dataset] = {"missing": str(path)}
            continue
        f = pq.ParquetFile(path)
        ix = f.schema_arrow.get_field_index("TransactTime")
        assert ix >= 0
        stats = [f.metadata.row_group(i).column(ix).statistics for i in range(f.num_row_groups)]
        # Metadata eliminates groups strictly before the cutoff.
        candidates = [i for i, s in enumerate(stats) if s is None or not s.has_min_max or s.max >= CUTOFF]
        counts = {"equal": Counter(), "after": Counter()}
        by_symbol = {"equal": Counter(), "after": Counter()}
        execution_types = {"equal": Counter(), "after": Counter()}
        examples = []
        null_times = sum(s.null_count or 0 for s in stats if s is not None)
        for i in candidates:
            columns = ["SecurityID", "TransactTime", "LocalTime", "ChannelNo", "ApplSeqNum"]
            if dataset == "mdl_6_36_0":
                columns.append("ExecType")
            table = f.read_row_group(i, columns=columns)
            times = table["TransactTime"]
            for label, mask in [("equal", pc.equal(times, CUTOFF)), ("after", pc.greater(times, CUTOFF))]:
                selected = table.filter(mask)
                for row in selected.to_pylist():
                    symbol = row["SecurityID"].strip()
                    counts[label][category(symbol)] += 1
                    by_symbol[label][symbol] += 1
                    execution_types[label][str(row.get("ExecType", "order"))] += 1
                    if len(examples) < 3 or (label == "after" and len(examples) < 10):
                        examples.append({"bucket": label, **row})
        distribution = {}
        for label in ["equal", "after"]:
            distribution[label] = {}
            for kind in ["stock", "etf", "other"]:
                values = sorted(n for symbol, n in by_symbol[label].items() if category(symbol) == kind)
                distribution[label][kind] = {
                    "symbols_with_records": len(values),
                    "records": sum(values),
                    "min_nonzero": min(values, default=0),
                    "median_nonzero": ((values[(len(values)-1)//2] + values[len(values)//2])/2 if values else 0),
                    "max": max(values, default=0),
                }
        result["files"][dataset] = {
            "path": str(path), "rows": f.metadata.num_rows,
            "row_groups": f.num_row_groups, "groups_read": candidates,
            "time_null_count_from_metadata": null_times,
            "max_transact_time": max((s.max for s in stats if s and s.has_min_max), default=None),
            "counts": counts, "distribution": distribution, "execution_types": execution_types,
            "examples": examples,
            "top_equal_symbols": by_symbol["equal"].most_common(5),
            "after_by_symbol": dict(by_symbol["after"]),
        }
    results.append(result)
print(json.dumps(results, ensure_ascii=False, indent=2))
